In [1]:
import pandas as pd

In [2]:
ruta_archivo_noticias = "./conjunto_noticias/conjunto_noticias_1.json"
df = pd.read_json(ruta_archivo_noticias)
df

,Link,Periódico,Fecha,Título,Subtítulo,Categoría,Contenido
0,https://www.20minutos.es/internacional/academi...,20minutos,2026-04-28,La Academia de Ciencias de EEUU retira el artí...,¡Barbacid y dos de las coautoras de su trabajo...,Internacional,La Academia de Ciencias de Estados Unidos ha d...
1,https://www.20minutos.es/internacional/estrech...,20minutos,2026-04-29,El estrecho Bab al Mandeb: todo sobre el otro ...,Por este estrecho pasa el 12% del comercio mar...,Internacional,La guerra en Irán ni acaba ni continúa. El con...
2,https://www.20minutos.es/internacional/un-trib...,20minutos,2026-04-28,Un tribunal de La Haya ordena el embargo de la...,"El inmueble, valorado en unos 10 millones de e...",Internacional,El Tribunal de Distrito de La Haya ha decretad...
3,https://www.20minutos.es/nacional/feijoo-acusa...,20minutos,2026-04-29,"Feijóo acusa a Sánchez de ""legislar contra los...",Tanto el PP como el PNV piden la dimisión de l...,Nacional,La presión política contra la ministra de Sani...
4,https://www.20minutos.es/nacional/gobierno-inv...,20minutos,2026-04-28,El Gobierno invitará a Delcy Rodríguez a la Cu...,"El ministro de Exteriores, José Manuel Albares...",Nacional,El Gobierno de España invitará a la presidenta...
...,...,...,...,...,...,...,...
58,https://www.rtve.es/noticias/20260424/muere-me...,RTVE,2026-04-24,Muere un menor de edad apuñalado por un joven ...,Los hechos han ocurrido en el barrio de Puente...,Nacional,La Policía Nacional investiga el homicidio de ...
59,https://www.rtve.es/noticias/20260423/mariano-...,RTVE,2026-04-23,Rajoy niega medidas para destruir pruebas que ...,El expresidente asegura que es “absolutamente ...,Nacional,Si hay un testigo del caso Kitchen que pueda m...
60,https://www.rtve.es/noticias/20260424/mami-mar...,RTVE,2026-04-24,'Mami' de Mario Banushi explora sin palabras l...,La última obra del joven director grecoalbanés...,Cultura,"para todas las mujeres que nos criaron "". El n..."
61,https://www.rtve.es/noticias/20260424/ahorcada...,RTVE,2026-04-24,'La ahorcada': un thriller español de terror s...,Miguel Ángel Lamata dirige esta película prota...,Cultura,"La ahorcada , es el nuevo largometraje de terr..."


In [3]:
df["Contenido"]

0     La Academia de Ciencias de Estados Unidos ha d...
1     La guerra en Irán ni acaba ni continúa. El con...
2     El Tribunal de Distrito de La Haya ha decretad...
3     La presión política contra la ministra de Sani...
4     El Gobierno de España invitará a la presidenta...
                            ...                        
58    La Policía Nacional investiga el homicidio de ...
59    Si hay un testigo del caso Kitchen que pueda m...
60    para todas las mujeres que nos criaron ". El n...
61    La ahorcada , es el nuevo largometraje de terr...
62    Tras contagiarnos su pasión por el cine y los ...
Name: Contenido, Length: 63, dtype: str

In [4]:
import re
import regex

## CARACTERES QUE HAY QUE ELIMINAR:
---

Tras una vista previa del texto guardado en el archivo JSON de `conjunto_noticias.json`, estos son los elementos del texto que se han detectado que deben o deberían eliminarse:
- **Comillas:** hay muchas formas diferentes que cada periódico utiliza para poner comillas, estas son: /""/, ««, '', "" y algunas más.  
- **Signos de interrogación y exclamación:** ¡! ¿?
- **Guiones:** --
- **URLs:**
- **Direcciones de correo y cuentas de twitter:** empiezan por @
- **Saltos de línea:** \n
- **Signos de puntuación:** , ; : .
- **Corchetes y paréntesis:** [] ()
- **Emojis**

In [ ]:
# FUNCIONES PARA LIMPIAR TEXTO:

# Quitar URLs
def remove_urls(texto):
    patron = re.compile(r"https?://\S+")
    texto = patron.sub("", texto)
    patron = re.compile(r"\b[A-Za-z0-9.-]+\.(com|es|net|org)(/[A-Za-z0-9._\-]+)*")
    return patron.sub("", texto)

# Quitar emojis
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags 
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           u"\U0001F900-\U0001F9FF"  # Emojis suplementarios
                           u"\U0001FA70-\U0001FAFF"  # Emojis más nuevos
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

# Quitar comillas
# Como hay tantos tipos de comillas y puede darse el caso de que haya incluso más de las que se han visto, se usa
# {Quotation_Mark} que es una lista con todos los códigos de todas las comillas posibles dentro del estándar UNICODE
# OTRA OPCIÓN: usar patron = r'[\"\'«»“”‘’]', pero si hay algún tipo de comilla que no está en la lista, no lo quita
def remove_comillas(texto):
    patron = r'\p{Quotation_Mark}'
    return regex.sub(patron, "", texto)

# Quitar saltos de línea
def remove_saltos_linea(texto):
    return texto.replace("\n", "").replace("\r", "")

# Signos de puntuación
# Hay un argumento opcional para quitar o no los puntos, para tener la opción de dividir el texto en frases
def remove_puntuacion_basica(texto, quitar_puntos):
    patron = r"[,;:]"
    if quitar_puntos:
        patron = r"[.,;:]"
    return re.sub(patron, "", texto)

# Menciones tipo Twitter (por ejemplo @usuario_123)
def remove_menciones(texto):
    patron = r"@[A-Za-z0-9_]+"
    return re.sub(patron, "", texto)

# Caracteres especiales y raros
def remove_esp(texto):
    patron = r"[#@]"
    return re.sub(patron, "", texto)

# Signos de exclamación e interrogación
def remove_int_excl(texto):
    texto = re.sub(r'[¡!¿?]+', '.', texto)
    texto = re.sub(r'\.+', '.', texto)
    return texto

# Paréntesis, corchetes y guiones
# Para los guiones, mejor sustituirlos por espacio, pues pueden aparecer en palabras compuestas (teórico-práctico) o para
# delimitar citas o información (-Una cita-). En el primer caso, elimnar el guion y no poner nada hace que se forme una
# palabra mal escrita, así que mejor poner espacio
def remove_delimitadores(texto):
    texto = re.sub(r'[\(\)\[\]]', '', texto)
    # Resulta que también hay varios tipos de guion diferentes, todos ellos se pueden quitar con la expresión regular
    # \p{Pd}
    return regex.sub(r"\p{Pd}", " ", texto)

# Después de haber eliminado muchos elementos del texto es normal que los espacios entre palabras ya no sean los correctos
# y que muchas palabras estén separadas por dos o más espacios, en vez de solo uno, así que debemos asegurar que 
# los espacios quedan bien
def clean_espacios(texto):
    # Reemplaza múltiples espacios por uno solo
    texto = re.sub(r"\s+", " ", texto)
    # Elimina espacios al inicio y al final
    return texto.strip()

In [162]:
# Vamos a hacer una función que ensamble todas las funciones de limpieza anteriores 
def Limpieza_texto(texto, minusculas=False, sign_excl_int=False, quitar_puntos=False):
    
    # Como algunas noticias no tienen subtítulo es posible que texto está vacío o sea none o NaN, así que hay que poner
    # un condicional
    if texto is None or pd.isna(texto):
        return ""

    # Puede ser que no queramos que el texto esté en minúsculas
    if minusculas:
        texto = texto.lower()
    
    # Puede ser que no queramos quitar los símbolos de exclamación e interrogación
    if sign_excl_int:
        texto = remove_int_excl(texto)

    texto = remove_urls(texto)
    texto = remove_menciones(texto)
    texto = remove_emoji(texto)
    texto = remove_comillas(texto)
    texto = remove_esp(texto)
    texto = remove_saltos_linea(texto)
    texto = remove_puntuacion_basica(texto, quitar_puntos=quitar_puntos)
    texto = remove_delimitadores(texto)

    return clean_espacios(texto)

In [ ]:
import os
import json

# - df: dataframe con el texto que vamos limpiar
# - ruta_origen: ruta local al archivo json donde se encuentra el dataframe con el texto que hay que limpiar
# - nombre_archivo: ruta donde queremos guardar el dataframe con el texto ya limpio
# - sustituir: en caso de que pueda existir un archivo con el mismo nombre para no correr el riesgo de sobreescribir y
# perder información importante, le ponemos que por defecto busque un nombre diferente

def Limpieza_y_guardado(ruta_origen, nombre_archivo, sustituir=False, 
                        minusculas=False, sign_excl_int=False, quitar_puntos=False):

    with open(ruta_origen, "r", encoding="utf-8") as f:
        df = pd.DataFrame(json.load(f))

    df['Contenido'] = df['Contenido'].apply(lambda x: Limpieza_texto(x, minusculas=minusculas, sign_excl_int=sign_excl_int, quitar_puntos=quitar_puntos))
    df['Título'] = df['Título'].apply(lambda x: Limpieza_texto(x, minusculas=minusculas, sign_excl_int=sign_excl_int, quitar_puntos=quitar_puntos))
    df['Subtítulo'] = df['Subtítulo'].apply(lambda x: Limpieza_texto(x, minusculas=minusculas, sign_excl_int=sign_excl_int, quitar_puntos=quitar_puntos))
    nombre_final = nombre_archivo
    # Si no se debe sustituir, buscamos un nombre alternativo
    if not sustituir:
        contador = 1
        base, ext = os.path.splitext(nombre_archivo)

        while os.path.exists(nombre_final):
            nombre_final = f"{base}_{contador}{ext}"
            contador += 1

    # Guardar JSON
    data = df.to_dict(orient="records")
    texto = json.dumps(data, ensure_ascii=False, indent=4)

    with open(nombre_final, "w", encoding="utf-8") as f:
        f.write(texto)

    return nombre_final


In [164]:
# Primera prueba
Limpieza_y_guardado(ruta_origen="./conjunto_noticias/conjunto_noticias_1.json",
                    nombre_archivo="./conjunto_noticias/conjunto_noticias_limpio.json",
                    minusculas=True)

'./conjunto_noticias/conjunto_noticias_limpio.json'

In [165]:
#with open("./conjunto_noticias/conjunto_noticias_limpio.json", "r", encoding="utf-8") as f:
#        df = pd.DataFrame(json.load(f))

#for elem in df["Contenido"]:
#        print("------------------------------------")
#        print(elem)

## Limpio + Sin Stopwords

---

In [30]:
#Cargamos librerías
import spacy
import nltk
from nltk.corpus import stopwords
import json
import os
import pandas as pd
import regex
import re

In [ ]:
# Cargamos herramientas de procesamiento

nltk.download('stopwords')               # Descargamos stopwords de NLTK
nlp = spacy.load("es_core_news_sm")      # Cargamos modelo spaCy para español para tokenizar
stop = set(stopwords.words('spanish'))   # Guardamos un set (más rápido de buscar que una lista) de stopwords en español en esa variable

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marti\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Para clickbait, algunas stopwords pueden ser informativas:

- por qué
- cómo
- qué
- este
- esto
- así
- no
- nunca
- nadie

Ejemplos:

- No creerás lo que pasó después
- Así usa tus impuestos el PSOE
- Este es el truco que nadie te cuenta

In [ ]:
# Creamos una versión con algunas stop words conservadas
# Sobretodo, interrogativas, negaciones, demostrativos
stop = set(stopwords.words('spanish'))

conservar = {

    # Negaciones
    "no", "nunca", "jamás", "nadie", "nada", "ningún", "ninguna",

    # Interrogativos / exclamativos
    "qué", "que",
    "cómo", "como",
    "cuál", "cuáles",
    "quién", "quiénes",
    "cuándo", "cuando",
    "dónde", "donde",

    # Demostrativos
    "este", "esta", "estos", "estas",
    "ese", "esa", "esos", "esas",
    "aquel", "aquella",
    "esto", "eso", "aquello",

    # Pronombres apelativos
    "tú", "tu", "usted", "ustedes",
    "te", "ti",

    # Intensificadores / énfasis
    "muy", "más", "mas",
    "tan", "tanto",
    "solo", "sólo",
    "incluso",

    # Conectores frecuentes en clickbait
    "porque", "por",
    "si",
    "aunque",

    # Auxiliares y verbos muy típicos
    "puede", "puedes",
    "debe", "debes",
    "vas", "van",
    "tiene", "tienen",

    # Expresiones típicas de titulares
    "así",
    "esto",
    "todo"
}

stop_filtradas = stop - conservar

In [ ]:
#probamos stop_filtradas
doc=nlp("No creerás lo que pasó después")
print([token.text for token in doc if token.text.lower() not in stop and token.is_alpha])
print([token.text for token in doc if token.text.lower() not in stop_filtradas and token.is_alpha])
doc=nlp("Así usan tus impuestos el PSOE")
print([token.text for token in doc if token.text.lower() not in stop and token.is_alpha])
print([token.text for token in doc if token.text.lower() not in stop_filtradas and token.is_alpha])
doc=nlp("Este es el truco que nadie te cuenta")
print([token.text for token in doc if token.text.lower() not in stop and token.is_alpha])
print([token.text for token in doc if token.text.lower() not in stop_filtradas and token.is_alpha])

['creerás', 'pasó', 'después']
['No', 'creerás', 'que', 'pasó', 'después']
['Así', 'usan', 'impuestos', 'PSOE']
['Así', 'usan', 'impuestos', 'PSOE']
['truco', 'nadie', 'cuenta']
['Este', 'truco', 'que', 'nadie', 'te', 'cuenta']


In [69]:
# Decidimos si queremos usar el filtro o no
conservar = False 

# Cargamos los datos
with open("conjunto_noticias/conjunto_noticias_limpio.json", "r", encoding="utf-8") as f:
    noticias = json.load(f)

In [ ]:
# Procesar cada noticia

for noticia in noticias:
    for campo in ["Título", "Subtítulo", "Contenido"]:
        if campo in noticia:
            doc = nlp(noticia[campo]) #tokenizamos el contenido de cada campo dentro de cada noticia
            # Filtrar stopwords
            if conservar:
                sinstopwords = [token.text for token in doc if token.text.lower() not in stop and token.is_alpha]
            else:
                sinstopwords = [token.text for token in doc if token.text.lower() not in stop_filtradas and token.is_alpha]

            noticia[campo] = " ".join(sinstopwords)

In [71]:
# Previsualizamos
noticias_df = pd.DataFrame(noticias)
noticias_df.loc[:,["Título", "Subtítulo", "Contenido"]].head()

,Título,Subtítulo,Contenido
0,academia ciencias eeuu retira artículo doctor ...,barbacid dos coautoras trabajo vasiliki liaki ...,academia ciencias unidos decidido retirar revi...
1,estrecho bab mandeb todo punto clave comercio ...,por este estrecho pasa comercio marítimo conte...,guerra irán acaba continúa conflicto unidos is...
2,tribunal ordena embargo sede instituto cervant...,inmueble valorado millones euros vendido prese...,tribunal distrito decretado embargo sede insti...
3,feijóo acusa sánchez legislar médicos mientras...,tanto pp como pnv piden dimisión ministra móni...,presión política ministra sanidad mónica garcí...
4,gobierno invitará delcy rodríguez cumbre ibero...,ministro exteriores josé manuel albares confir...,gobierno españa invitará presidenta encargada ...


In [72]:
# Comparamos con el original, necesario cargar df (Sección de Juan, segundo chunk)
df.loc[:,["Título", "Subtítulo", "Contenido"]].head()

,Título,Subtítulo,Contenido
0,La Academia de Ciencias de EEUU retira el artí...,¡Barbacid y dos de las coautoras de su trabajo...,La Academia de Ciencias de Estados Unidos ha d...
1,El estrecho Bab al Mandeb: todo sobre el otro ...,Por este estrecho pasa el 12% del comercio mar...,La guerra en Irán ni acaba ni continúa. El con...
2,Un tribunal de La Haya ordena el embargo de la...,"El inmueble, valorado en unos 10 millones de e...",El Tribunal de Distrito de La Haya ha decretad...
3,"Feijóo acusa a Sánchez de ""legislar contra los...",Tanto el PP como el PNV piden la dimisión de l...,La presión política contra la ministra de Sani...
4,El Gobierno invitará a Delcy Rodríguez a la Cu...,"El ministro de Exteriores, José Manuel Albares...",El Gobierno de España invitará a la presidenta...


In [ ]:
# Guardar resultado
if conservar:
    with open("conjunto_noticias/conjunto_noticias_procesado_1_2.json", "w", encoding="utf-8") as f:
        json.dump(noticias, f, ensure_ascii=False, indent=2)
else:
    with open("conjunto_noticias/conjunto_noticias_procesado_1_2_filter.json", "w", encoding="utf-8") as f:
        json.dump(noticias, f, ensure_ascii=False, indent=2)

## Limpio + Lematizado + Sin Stopwords

---

In [1]:
import spacy
import nltk
from nltk.corpus import stopwords
import json
nltk.download('stopwords')
nlp = spacy.load("es_core_news_sm")
stop = stopwords.words('spanish')

with open("conjunto_noticias/conjunto_noticias_limpio.json", "r", encoding="utf-8") as f:
    noticias = json.load(f)

# Procesar cada noticia
for noticia in noticias:
    for campo in ["Título", "Subtítulo", "Contenido"]:
        if campo in noticia:
            doc = nlp(noticia[campo])
            # Lematizar y filtrar stopwords
            lemmas = [token.lemma_ for token in doc if token.text.lower() not in stop and token.is_alpha]
            noticia[campo] = " ".join(lemmas)

# Guardar resultado
with open("conjunto_noticias/conjunto_noticias_procesado_1_3.json", "w", encoding="utf-8") as f:
    json.dump(noticias, f, ensure_ascii=False, indent=2)

/home/javier/miniconda3/envs/NLP/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
[nltk_data] Downloading package stopwords to /home/javier/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Limpio + Lematizado + Con Stopwords

---

In [ ]:
import spacy
import nltk
from nltk.corpus import stopwords
import json
nltk.download('stopwords')
nlp = spacy.load("es_core_news_sm")
stop = stopwords.words('spanish')

with open("conjunto_noticias/conjunto_noticias_limpio.json", "r", encoding="utf-8") as f:
    noticias = json.load(f)

# Procesar cada noticia
for noticia in noticias:
    for campo in ["Título", "Subtítulo", "Contenido"]:
        if campo in noticia:
            doc = nlp(noticia[campo])
            # Lematizar y filtrar stopwords
            lemmas = [token.lemma_ for token in doc if token.is_alpha]
            noticia[campo] = " ".join(lemmas)

# Guardar resultado
with open("conjunto_noticias/conjunto_noticias_procesado_1_4.json", "w", encoding="utf-8") as f:
    json.dump(noticias, f, ensure_ascii=False, indent=2)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\clara\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
